# SynEHRgy Results Analysis

This notebook provides comprehensive evaluation of synthetic EHR data generation, including:
- **Fidelity**: N-gram analysis for ICD codes, correlation matrices for time series
- **Utility**: Downstream task performance (mortality prediction, phenotyping)
- **Privacy**: Distance-based metrics for both ICD codes and time series

## Setup

In [1]:
# Jupyter notebook configuration
%load_ext autoreload
%autoreload 2

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
# Standard library imports
import os
import pickle
import random
from collections import defaultdict

# Third-party imports
import dill
import numpy as np
import pandas as pd
from tqdm import tqdm
from omegaconf import DictConfig, OmegaConf

# Visualization imports
import matplotlib.pyplot as plt
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go

# Scikit-learn imports
from sklearn.metrics import (
    confusion_matrix, 
    ConfusionMatrixDisplay,
    r2_score,
    roc_auc_score
)

# Statistical imports
from scipy.stats import pearsonr, wasserstein_distance, gaussian_kde
from scipy.spatial.distance import jensenshannon

# OpenTSNE for dimensionality reduction
from openTSNE import TSNE

# External library for string distance
import Levenshtein

# Project-specific imports
from synehrgy.utils import *

/home/hokarami/.conda/envs/synehrgy_results/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
# save to pickle
with open("token_lengths.pkl", "rb") as f:
    token_lengths = pickle.load(f)
# define custom bin edges
bin_edges = np.array([0, 64, 256, 512, 1024, 2048, 4096, 8192, 20000])

# use equal binsize=50
bin_edges = np.arange(0, 20001, 200)

# compute histogram
bin_heights, _ = np.histogram(token_lengths, bins=bin_edges)

# normalize to probabilities
bin_heights = bin_heights.astype(float) / bin_heights.sum()

# cumulative heights
bin_heights = np.cumsum(bin_heights)

# finite bin edges for display
finite_edges = np.where(np.isinf(bin_edges), np.nan, bin_edges)
bin_centers = 0.5 * (finite_edges[:-1] + np.nan_to_num(finite_edges[1:], nan=finite_edges[-2]*2))
bin_widths = np.diff(np.nan_to_num(finite_edges, nan=finite_edges[-2]*2))

# build figure
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=bin_centers,
        y=bin_heights,
        width=bin_widths,
        marker_color="blue",
        name="Token Length Distribution",
    )
)

# update layout
fig.update_layout(
    title=f"Token Length Distribution ({len(token_lengths)} samples)",
    xaxis_title="Token Length",
    yaxis_title="Count",
    bargap=0.1,
)




fig.show()


usage = {}
for n_ctx in [256, 512, 1024, 2048, 4096, 8192, 8192*2]:
    usage[n_ctx] = sum([x if x < n_ctx else n_ctx for x in token_lengths])
    usage[n_ctx] /= sum(token_lengths)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=list(usage.keys()),
        y=list(usage.values()),
        mode="lines+markers",
        name="Token Usage",
    )
)
fig.update_layout(
    title="Token Usage by Context Length",
    xaxis_title="Context Length",
    yaxis_title="Usage",
)


In [31]:
path="/home/hokarami/code/SynEHRgy/data/synthetic/v8-gpt3-var+quantDataset.pkl"
# path="/home/hokarami/code/SynEHRgy/data/synthetic/v6-gpt3Dataset.pkl"

data = pickle.load(open(path, "rb"))


In [27]:
for d in data:
    # print(d['covars'])
    if len(d['covars'])>0:
        print((d['covars'][0]))
    # if d['covars'][0]==[]:



([], [])
([], [])
[]
([42], [0])
[]
([], [])
([], [])
([], [])
([42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
[]
([42, 41], [0, 14])
([42, 42, 42, 42], [1, 1, 1, 1])
([42, 42], [1, 0])
[]
([], [])
([42, 42], [1, 0])
([42, 41, 42], [0, 14, 0])
[]
([], [])
([], [])
([42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42], [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
([42], [0])
([], [])
([], [])
([], [])
([42], [1])
([42], [0])
([], [])
([], [])
[]
([], [])
([], [])
[]
([42, 42, 42, 42, 42, 42, 42, 42, 42, 42], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
[]
([42, 42], [1, 1])
([42], [1])
([], [])
([], [])


In [14]:
data[0]
for k,v in data[0].items():
    print(k, len(v))
data[0]['covars'][0]

{'covars': [([41, 42], [12, 1]), []],
 'codes': [[1,
   2,
   66,
   16,
   94,
   97,
   8,
   3,
   771,
   3751,
   3640,
   3617,
   3607,
   3628,
   3694],
  []],
 'ts': [[([1, 2, 3, 4, 17, 20, 25, 29, 36, 37],
    [2, 1, 3, 1, 7, 7, 6, 9, 3, 8],
    [0]),
   ([17, 20, 25, 29, 36, 37], [6, 5, 4, 9, 4, 7], [0]),
   ([17, 20, 25, 29, 36, 37], [6, 3, 3, 9, 8, 7], [0]),
   ([17, 20, 25, 29, 36, 37], [5, 1, 3, 9, 5, 7], [0]),
   ([5,
     6,
     7,
     8,
     9,
     11,
     12,
     13,
     14,
     15,
     16,
     19,
     21,
     22,
     26,
     31,
     32,
     33,
     34,
     35,
     39],
    [0, 2, 1, 6, 0, 6, 0, 0, 5, 4, 1, 5, 7, 7, 2, 4, 4, 7, 4, 7, 3],
    [0]),
   ([17, 20, 25, 29, 36, 37], [6, 1, 2, 9, 5, 6], [0]),
   ([1, 2, 3, 4, 17, 20, 25, 29, 36, 37], [2, 1, 3, 1, 6, 1, 3, 8, 4, 6], [0]),
   ([17, 20, 25, 29, 36, 37], [6, 1, 2, 9, 4, 6], [0]),
   ([17, 20, 25, 29, 36, 37], [6, 1, 3, 8, 5, 6], [0]),
   ([17, 20, 25, 29, 36, 37, 38], [6, 1, 2, 9, 5, 6, 0], 

covars 2
codes 2
ts 2
labels_phe 2
labels_ihm 2


([41, 42], [12, 1])

In [ ]:
len(data)
data[0].keys()

for d in data:
    if len(d['ts'][0]) != 0:
        
        print(d['ts'][0])

In [33]:
data[1]

{'covars': [([41, 42], [11, 1])],
 'codes': [[181, 124, 92, 775, 89, 18, 275, 275, 89, 29, 49, 41, 3619, 3599]],
 'ts': [[([17, 25, 37], [8, 9, 9], [2]),
   ([20, 36], [5, 4], [0]),
   ([29], [5], [0]),
   ([20, 29, 36, 38], [5, 5, 4, 1], [0]),
   ([17, 20, 25, 29, 36, 37], [9, 5, 9, 5, 4, 9], [0]),
   ([1, 2, 4], [2, 1, 1], [0]),
   ([5,
     6,
     7,
     8,
     9,
     10,
     11,
     12,
     13,
     14,
     15,
     16,
     19,
     21,
     22,
     24,
     26,
     27,
     28,
     31,
     32,
     33,
     34,
     35,
     39],
    [0,
     5,
     3,
     7,
     0,
     0,
     6,
     0,
     1,
     5,
     3,
     3,
     9,
     6,
     6,
     1,
     4,
     4,
     6,
     4,
     0,
     8,
     5,
     5,
     1],
    [0]),
   ([17, 20, 25, 29, 36, 37], [9, 5, 9, 6, 3, 9], [0]),
   ([23], [0], [0]),
   ([11, 12, 15, 16, 19], [6, 0, 3, 3, 9], [0]),
   ([17, 19, 20, 25, 29, 36, 37], [9, 9, 4, 8, 3, 3, 9], [0]),
   ([20, 29, 36], [4, 3, 3], [0]),
   ([20, 29

In [15]:
pp="/home/hokarami/code/SynEHRgy/data/processed/mimic3-v2/valDiscDataset_uniform_v1.pkl"

data = pickle.load(open(pp, "rb"))
len(data)
data[0].keys()

5155

dict_keys(['covars', 'codes', 'ts', 'labels_phe', 'labels_ihm', 'horizons'])

In [20]:
data[4]

{'covars': [([41, 42], [4, 1]), ([41, 42], [4, 1])],
 'codes': [[251, 40, 373, 44, 42, 0, 3671, 3670, 3989, 3604],
  [317, 2, 4, 9, 10, 0, 185, 127, 20]],
 'ts': [[([19, 21, 22, 30, 33, 40], [5, 7, 7, 8, 5, 9], [1]),
   ([19, 21, 22, 30, 33, 40], [5, 7, 7, 7, 9, 9], [1]),
   ([19, 21, 22, 30, 33, 40], [5, 7, 7, 5, 8, 9], [1]),
   ([32, 34], [2, 0], [1]),
   ([19, 21, 22, 30, 33, 36, 40], [3, 7, 6, 5, 6, 2, 9], [0]),
   ([17, 20, 25, 29, 36, 37, 38], [6, 5, 1, 9, 4, 5, 2], [0]),
   ([13, 15, 16, 21, 32, 34], [0, 8, 0, 8, 2, 5], [0]),
   ([17, 20, 25, 29, 36, 37, 38], [6, 5, 2, 9, 4, 5, 2], [0]),
   ([19, 30, 33, 40], [2, 6, 7, 9], [0]),
   ([1, 2, 3, 4, 17, 20, 25, 29, 36, 37, 38],
    [0, 3, 7, 0, 7, 4, 3, 9, 2, 6, 3],
    [0]),
   ([21, 22, 29], [8, 8, 0], [0]),
   ([17, 20, 25, 29, 36, 37, 38], [7, 4, 4, 9, 2, 6, 3], [0]),
   ([17, 20, 25, 29, 36, 37, 38], [7, 4, 4, 9, 2, 7, 4], [0]),
   ([17, 20, 25, 29, 36, 37, 38], [6, 3, 1, 9, 3, 5, 3], [0]),
   ([1, 2, 3, 4], [1, 1, 6, 0], [0]),

In [29]:

from datasets import Dataset



hf_dataset = Dataset.from_list(data[:])

# hf_dataset.save_to_disk("/home/hokarami/code/SynEHRgy/data/processed/mimic3-v2/hf_valDiscDataset_uniform_v1")


/home/hokarami/.conda/envs/paper2025/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AttributeError: 'numpy.ndarray' object has no attribute 'get'

In [30]:
data[0]

array([4716, 4669, 4673, ..., 4723, 4723, 4723])

In [44]:
from synehrgy.models import SynEHRgy
model_path = f"./saved_models/v6-gpt3"

trainer = SynEHRgy.from_pretrained(model_path,
                                    #    train_dataset=datasets['train'],
                                    #    eval_dataset=datasets['test'],
                                       )


2025-10-30 16:38:54.590573: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-30 16:38:54.795178: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761842334.900974   34674 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761842334.943287   34674 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1761842335.172983   34674 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Loading from: ./saved_models/v6-gpt3/checkpoint-41


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model size: 9.46M


In [ ]:
trainer

In [31]:
len(hf_dataset)
hf_dataset

for feature, dtype in hf_dataset.features.items():
    print(f"\tFeature: {feature}, Type: {dtype}")

hf_dataset['codes']

10

Dataset({
    features: ['covars', 'codes', 'ts', 'labels_phe', 'labels_ihm', 'horizons'],
    num_rows: 10
})

	Feature: covars, Type: List(List(List(Value('int64'))))
	Feature: codes, Type: List(List(Value('int64')))
	Feature: ts, Type: List(List(List(List(Value('int64')))))
	Feature: labels_phe, Type: List(List(Value('int64')))
	Feature: labels_ihm, Type: List(Value('int64'))
	Feature: horizons, Type: List(List(Value('int64')))


Column([[[1273, 44, 2, 3671, 3752, 3604], [731, 2, 1097, 260, 2026, 1154, 185, 326, 897, 3683, 3681, 3611, 3718, 3871, 3672]], [[2123, 7, 854, 230, 188, 640, 4, 623, 0, 14, 87, 51, 82, 3742, 3600, 3599, 3608, 3663]], [[325, 126, 10, 151]], [[31, 1, 18, 674, 16, 15, 3, 5, 42, 3628, 3615]], [[7, 5, 0, 6, 204, 121, 3602, 3613, 3637, 3600]]])

In [ ]:
## Configuration

In [ ]:
# ==================== GLOBAL CONFIGURATION ====================

# Output configuration
OUTPUT_DIR = 'RESULTS/run0'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Data paths
PATH_REAL = "data/processed/mimic3/{SPLIT}DiscDataset.pkl"
PATH_TS = "data/processed/mimic3/{SPLIT}TS.csv"
PATH_META = "data/processed/mimic3/metadata2.pkl"
PATH_SYN = "data/synthetic/{RUN_NAME}Dataset.pkl"
PATH_TS_CACHE = "data/SynTS"  # Cache for de-discretized time series data

# Model/run names
SYNEHRGY_NAME = "synehrgy-mimic-v2"
PEHR_NAME = "pehrBig"
HALO_NAME = 'halo-icd-big'

# Run configurations
RUN_NAMES_ICD = ['synehrgy', 'HALO', 'pehr']  # For ICD code analysis
RUN_NAMES = [SYNEHRGY_NAME, 'timehr', 'rtsgan-phe']  # For time series analysis

# Continuous variables (36 total)
CONT_VARS = [
    'Alanine aminotransferase', 'Albumin', 'Alkaline phosphate', 'Anion gap',
    'Asparate aminotransferase', 'Basophils', 'Bicarbonate', 'Bilirubin',
    'Blood urea nitrogen', 'Calcium', 'Chloride', 'Creatinine',
    'Diastolic blood pressure', 'Fraction inspired oxygen', 'Glucose', 'Heart Rate',
    'Hematocrit', 'Hemoglobin', 'Lactate', 'Lymphocytes', 'Mean blood pressure',
    'Mean corpuscular volume', 'Monocytes', 'Neutrophils', 'Oxygen saturation',
    'Partial pressure of carbon dioxide', 'Phosphate', 'Platelets', 'Potassium',
    'Prothrombin time', 'Red blood cell count', 'Respiratory rate',
    'Systolic blood pressure', 'Temperature', 'White blood cell count', 'pH'
]

# Label configurations
N_PHE_LABELS = 25  # Number of phenotype labels
COL_LABELS = [f'label_phe_{i}' for i in range(N_PHE_LABELS)] + ['label_ihm']

# Analysis parameters
N_TOP_NGRAMS = 1000000  # Top N n-grams to consider in analysis
N_SAMPLES_PRIVACY = 5000  # Number of samples for privacy analysis
N_SAMPLES_UTILITY = 5000  # Number of samples for utility analysis
TIME_WINDOW_HOURS = 48  # Time window for time series analysis

print(f"✓ Configuration loaded")
print(f"  - Output directory: {OUTPUT_DIR}")
print(f"  - Continuous variables: {len(CONT_VARS)}")
print(f"  - ICD models: {RUN_NAMES_ICD}")
print(f"  - Time series models: {RUN_NAMES}")

In [ ]:
# Load metadata
metadata = pickle.load(open(PATH_META, 'rb'))

# Extract metadata components
indexToCode = metadata['idToCode']
codeToIndex = metadata['codeToId']
possibleValues = metadata['possibleValues']
isCategorical = metadata['isCategorical']
discretization = metadata['discretization']
codeToId = metadata['codeToId']
ts_info = metadata['ts_info']
token2id = metadata['token2id']
var2id = metadata['var2id']
id2var = {v: k for k, v in var2id.items()}
N_WORDS = metadata['vocab_size']['codes']

print(f"✓ Metadata loaded successfully")

## Helper Functions

In [ ]:
# ==================== DATA LOADING HELPERS ====================

def sample_from_discretization(i, name):
    """Uniform sampling from discretized intervals."""
    return random.uniform(discretization[name][i], discretization[name][i + 1])


def get_df_ts_covars(k, dataset, force_compute=False, save=True):
    """
    Convert discretized time series data to DataFrame with continuous and categorical values.
    
    Args:
        k: Dataset name/key
        dataset: Input dataset
        force_compute: Force recomputation even if cached
        save: Save results to disk
        
    Returns:
        (dfs_ts, dfs_covar): Time series and covariate DataFrames
    """
    cache_ts = f"{PATH_TS_CACHE}/df-ts-{k}.pkl"
    cache_static = f"{PATH_TS_CACHE}/df-static-{k}.pkl"
    
    # Try to load from cache
    if os.path.exists(cache_ts) and not force_compute:
        print(f"Loading cached data for {k}")
        with open(cache_ts, 'rb') as f:
            dfs_ts = pickle.load(f)
        with open(cache_static, 'rb') as f:
            dfs_covar = pickle.load(f)
        return dfs_ts, dfs_covar
    
    print(f"Processing time series data for {k}...")
    sub_dataset = dataset[:]
    all_dfs = []
    all_covars = []
    
    for i_p, patient in tqdm(enumerate(sub_dataset), total=len(sub_dataset), desc=k):
        # Convert covariates
        if len(patient['covars']) == 0:
            continue
            
        covars = patient['covars'][0]
        labels = patient['labels_phe'][0]
        temp_covar = {'id': i_p}
        
        # Add labels
        for i, label in enumerate(labels):
            temp_covar[f'label_phe_{i}'] = label
        temp_covar['label_ihm'] = patient['labels_ihm'][0]
        
        # Add covariate values
        if len(covars) > 0:
            for covar_id, covar_value in zip(covars[0], covars[1]):
                name = id2var[covar_id]
                if isCategorical[name]:
                    covar_value = possibleValues[name][covar_value]
                else:
                    covar_value = sample_from_discretization(covar_value, name)
                temp_covar[name] = covar_value
        
        all_covars.append(temp_covar)
        
        # Process time series (first admission only)
        for admission in patient['ts'][:1]:
            prev_time = 0
            ts_data = []
            
            for measurement in admission:
                if measurement[1] == []:
                    continue
                
                indices = measurement[0]
                values = measurement[1]
                time_gap = measurement[2][0]
                
                timestamp = prev_time + sample_from_discretization(time_gap, 'Hours')
                prev_time = timestamp
                
                empty_rec = {'id': i_p, 'Hours': timestamp}
                empty_rec.update({name: np.nan for name in list(ts_info.keys())})
                
                for idx, value in zip(indices, values):
                    name = id2var[idx]
                    if isCategorical[name]:
                        empty_rec[name] = value
                    else:
                        try:
                            empty_rec[name] = sample_from_discretization(value, name)
                        except Exception as e:
                            print(f"Error processing {name}, {value}: {e}")
                
                ts_data.append(empty_rec)
            
            all_dfs.append(pd.DataFrame(ts_data))
    
    # Combine dataframes
    dfs_ts = pd.concat(all_dfs)
    dfs_covar = pd.DataFrame(all_covars)
    
    # Keep only common IDs
    print(f"Before filtering: {dfs_ts.id.nunique()} TS patients, {dfs_covar.id.nunique()} static patients")
    common_ids = pd.merge(dfs_ts[['id']], dfs_covar[['id']], on='id')
    dfs_ts = dfs_ts[dfs_ts['id'].isin(common_ids['id'])]
    dfs_covar = dfs_covar[dfs_covar['id'].isin(common_ids['id'])]
    print(f"After filtering: {dfs_ts.id.nunique()} TS patients, {dfs_covar.id.nunique()} static patients")
    
    # Save to cache
    if save:
        os.makedirs(PATH_TS_CACHE, exist_ok=True)
        with open(cache_ts, 'wb') as f:
            pickle.dump(dfs_ts, f)
        with open(cache_static, 'wb') as f:
            pickle.dump(dfs_covar, f)
        print(f"Cached data saved for {k}")
    
    return dfs_ts, dfs_covar


# PromptEHR-specific data class
class Voc(object):
    """Vocabulary class for PromptEHR data format."""
    def __init__(self):
        self.idx2word = {}
        self.word2idx = {}

    def add_sentence(self, sentence):
        for word in sentence:
            if word not in self.word2idx:
                self.idx2word[len(self.word2idx)] = word
                self.word2idx[word] = len(self.word2idx)

In [ ]:
# ==================== N-GRAM ANALYSIS HELPERS ====================

def compute_ngram_dict(sequences, n):
    """
    Compute n-gram counts for sequences.
    
    Args:
        sequences: List of sequences
        n: N-gram size
        
    Returns:
        Dictionary of n-gram counts
    """
    ngram_dict = defaultdict(int)
    
    for seq in sequences:
        for i in range(len(seq) - n + 1):
            ngram = tuple(seq[i:i+n])
            ngram_dict[ngram] += 1
    
    return dict(ngram_dict)


def compute_bigram_seq(sequences):
    """
    Compute sequential bi-grams across patient visits.
    
    Args:
        sequences: List of (visit1, visit2) tuples
        
    Returns:
        Dictionary of normalized bi-gram probabilities
    """
    ngram_dict = defaultdict(int)
    total_w1 = {}
    
    for seq in sequences:
        sub1, sub2 = seq[0], seq[1]
        for i in range(len(sub1)):
            for j in range(len(sub2)):
                bigram = (sub1[i], sub2[j])
                ngram_dict[bigram] += 1
                total_w1[sub1[i]] = total_w1.get(sub1[i], 0) + 1
    
    # Normalize by first word occurrence
    ngram_dict = {k: v / total_w1[k[0]] for k, v in ngram_dict.items()}
    return dict(ngram_dict)


# ==================== STATISTICAL HELPERS ====================

def compute_jsd(d1, d2):
    """
    Compute Jensen-Shannon divergence between two distributions using KDE.
    
    Args:
        d1, d2: Data arrays
        
    Returns:
        JSD value or NaN if insufficient data
    """
    if len(d1) < 10 or len(d2) < 10:
        return np.nan
    
    kde1 = gaussian_kde(d1)
    kde2 = gaussian_kde(d2)
    
    max_val = max(max(d1), max(d2))
    x = np.linspace(0, max_val, 1000)
    p1 = kde1(x)
    p2 = kde2(x)
    
    return jensenshannon(p1, p2)


def prettify_metrics(all_metrics):
    """
    Convert list of metric dictionaries to formatted DataFrame.
    
    Args:
        all_metrics: List of metric dictionaries
        
    Returns:
        DataFrame with mean values (standard deviation)
    """
    metric_names = list(pd.DataFrame(all_metrics[0]).index)
    columns = list(pd.DataFrame(all_metrics[0]).columns)
    
    mat = np.stack([pd.DataFrame(m).values for m in all_metrics])
    
    df_mean = pd.DataFrame(np.mean(mat, axis=0), columns=columns, index=metric_names).round(3).T
    df_std = pd.DataFrame(np.std(mat, axis=0), columns=columns, index=metric_names).round(3).T
    
    # Combine mean and std
    df = df_mean.astype(str)  # + " (" + df_std.astype(str) + ")"
    return df


# ==================== PRIVACY ANALYSIS HELPERS ====================

def hamming_distance(seq1, seq2):
    """
    Compute set-based distance between two sequences.
    
    Args:
        seq1, seq2: Input sequences
        
    Returns:
        Number of different elements
    """
    return len(set(seq1) ^ set(seq2))


def pairwise_hamming_distance(dataset1, dataset2):
    """
    Compute pairwise Hamming distance matrix between two datasets.
    
    Args:
        dataset1, dataset2: Lists of sequences
        
    Returns:
        Distance matrix (len1 x len2)
    """
    len1, len2 = len(dataset1), len(dataset2)
    D = np.zeros((len1, len2), dtype=int)
    
    for i in tqdm(range(len1), desc="Computing distances"):
        for j in range(len2):
            D[i, j] = hamming_distance(dataset1[i], dataset2[j])
    
    return D


def comp_acc(d1, d2):
    """Compute balanced accuracy for distance comparison."""
    acc = 0.5 * (sum(d1 < d2) / len(d1) + sum(d1 > d2) / len(d1))
    return acc


# ==================== VISUALIZATION HELPERS ====================

def plot_tsne(data: dict, N: int = 10000) -> go.Figure:
    """
    Create t-SNE visualization for multiple datasets.
    
    Args:
        data: Dictionary of dataset arrays
        N: Number of samples per dataset
        
    Returns:
        Plotly Figure object
    """
    X = []
    for k in data.keys():
        random_indices = np.random.choice(data[k].shape[0], size=N, replace=False)
        X.append(data[k][random_indices, :])
    
    X = np.concatenate(X, axis=0)
    
    tsne = TSNE(n_components=2, perplexity=30, learning_rate=10, n_jobs=4)
    X_tsne = tsne.fit(X)
    
    fig_tsne = go.Figure()
    
    for k in data.keys():
        _ = fig_tsne.add_trace(
            go.Scatter(x=X_tsne[:N, 0], y=X_tsne[:N, 1], mode="markers", name=k)
        )
        X_tsne = X_tsne[N:]
    
    fig_tsne.update_traces(marker=dict(opacity=0.75, size=5))
    
    return fig_tsne

print("✓ Helper functions loaded")

# Data Loading

## ICD Code Datasets

In [ ]:
# Define synthetic data paths
dict_path_syn = {
    'synehrgy': PATH_SYN.format(RUN_NAME=SYNEHRGY_NAME),
    'halo': PATH_SYN.format(RUN_NAME=HALO_NAME),
    'pehr': PATH_SYN.format(RUN_NAME=PEHR_NAME),
}

In [ ]:
# Load ICD datasets (real + synthetic)
print("Loading ICD code datasets...")

datasets_icd = {
    'train': pickle.load(open(PATH_REAL.format(SPLIT='train'), 'rb')),
    'test': pickle.load(open(PATH_REAL.format(SPLIT='test'), 'rb')),
    'synehrgy': pickle.load(open(dict_path_syn['synehrgy'], 'rb')),
    'HALO': pickle.load(open(dict_path_syn['halo'], 'rb')), 
}

print(f"✓ Loaded {len(datasets_icd)} ICD datasets")

In [ ]:
# Load PromptEHR data (special format)
print("Loading PromptEHR data...")

temp = dill.load(open(PATH_SYN.format(RUN_NAME=PEHR_NAME), 'rb'))

data = []
for v in tqdm(temp['visit'], desc="Processing PromptEHR"):
    code_diags = [codeToIndex[temp['voc']['diag_voc'].idx2word[x]] for x in v[0][0]]
    code_procs = [codeToIndex[temp['voc']['pro_voc'].idx2word[x]] for x in v[0][1]]
    
    data.append({
        'visits': [code_diags + code_procs],
        'labels': np.zeros(N_PHE_LABELS)
    })

datasets_icd['pehr'] = data
print(f"✓ Loaded PromptEHR with {len(data)} patients")

In [ ]:
# Extract ICD codes only (HALO and PEHR already have ICD codes only)
print("Extracting ICD codes from datasets...")

datasets_icd['synehrgy'] = [
    {'visits': p['codes'], 'labels': p['labels_phe']} 
    for p in datasets_icd['synehrgy']
]

datasets_icd['train'] = [
    {'visits': p['codes'], 'labels': p['labels_phe']} 
    for p in datasets_icd['train']
]

datasets_icd['test'] = [
    {'visits': p['codes'], 'labels': p['labels_phe']} 
    for p in datasets_icd['test']
]

print("✓ ICD codes extracted")

In [ ]:
# Remove empty patients from ICD datasets
print("[INFO] Removing empty patients from ICD datasets")

for k, dataset in datasets_icd.items():
    len_before = len(dataset)
    datasets_icd[k] = [x for x in dataset if len(x['visits']) > 0]
    print(f"  {k}: {len_before} → {len(datasets_icd[k])}")

## Time Series Datasets

In [ ]:
# Load real data splits
print("Loading real time series datasets...")

datasets = {
    'train': pickle.load(open(PATH_REAL.format(SPLIT='train'), 'rb')),
    'test': pickle.load(open(PATH_REAL.format(SPLIT='test'), 'rb')),
    'val': pickle.load(open(PATH_REAL.format(SPLIT='val'), 'rb')),
}

print(f"✓ Loaded {len(datasets)} real datasets")

In [ ]:
# Load synthetic time series data
print("Loading synthetic time series datasets...")

for run in tqdm(RUN_NAMES, desc="Loading synthetic data"):
    try:
        datasets[run] = pickle.load(open(PATH_SYN.format(RUN_NAME=run), 'rb'))
    except Exception as e:
        print(f"  Error loading {run}: {e}")
        datasets[run] = None

print(f"✓ Loaded {sum([1 for v in datasets.values() if v is not None])} total datasets")

In [ ]:
# Remove empty patients from time series datasets
print("[INFO] Removing empty patients from time series datasets")

for k, dataset in datasets.items():
    if dataset is None:
        print(f"  {k}: Not found")
        continue
    
    try:
        len_before = len(dataset)
        datasets[k] = [x for x in dataset if len(x['codes']) > 0]
        print(f"  {k}: {len_before} → {len(datasets[k])}")
    except Exception as e:
        print(f"  {k}: Error - {e}") 
        datasets[k] = None

In [ ]:
# Convert to DataFrames with continuous values
print("[INFO] Converting to DataFrames (this may take some time)...")

datasets_df_ts = {}
datasets_df_static = {}

for k in ['train', 'test', 'val'] + RUN_NAMES:

    
    print(f"Processing {k}...")
    
    if k in ['train', 'test', 'val']:
        # Load pre-processed CSV files for real data
        datasets_df_ts[k] = pd.read_csv(PATH_TS.format(SPLIT=k)).rename(
            columns={"RecordID": 'id', 'Time': 'Hours'}
        )
        
        labels = ['label_ihm'] + [f'label_phe_{i}' for i in range(N_PHE_LABELS)]
        datasets_df_static[k] = datasets_df_ts[k][['id', 'Age', 'Gender'] + labels]\
            .groupby('id').first().reset_index()
        datasets_df_ts[k] = datasets_df_ts[k].drop(columns=['Age', 'Gender'] + labels)
        
    elif 'rtsgan' in k:
        # Load pre-processed files for RTSGAN
        datasets_df_ts[k] = pd.read_csv(f"{PATH_TS_CACHE}/df-ts-{k}.csv").rename(
            columns={"RecordID": 'id', 'Time': 'Hours'}
        )
        datasets_df_static[k] = pd.read_csv(f"{PATH_TS_CACHE}/df-static-{k}.csv").rename(
            columns={"RecordID": 'id', 'Label': 'label_ihm'}
        ).drop(columns=['seq_len'])
        
    else:
        # Process other synthetic data
        dfs_ts, dfs_covar = get_df_ts_covars(k, datasets[k])
        datasets_df_static[k] = dfs_covar
        datasets_df_ts[k] = dfs_ts
    
    # Fix label naming inconsistencies
    if 'phe_0' in datasets_df_static[k].columns:
        print(f"  Fixing labels for {k}")
        datasets_df_static[k].rename(
            columns={f'phe_{i}': f'label_phe_{i}' for i in range(N_PHE_LABELS)}, 
            inplace=True
        )
    
    if 'Label' in datasets_df_static[k].columns:
        print(f"  Fixing label_ihm for {k}")
        datasets_df_static[k].rename(columns={'Label': 'label_ihm'}, inplace=True)

print("✓ DataFrame conversion complete")

In [ ]:
# Filter to first 48 hours only
print(f"[INFO] Filtering time series to first {TIME_WINDOW_HOURS} hours")

for k in datasets_df_ts.keys():
    before_count = len(datasets_df_ts[k])
    datasets_df_ts[k] = datasets_df_ts[k][datasets_df_ts[k]['Hours'] < TIME_WINDOW_HOURS]
    after_count = len(datasets_df_ts[k])
    print(f"  {k}: {before_count} → {after_count} measurements")

print("✓ Time filtering complete")

In [ ]:
# Create time series embeddings
print("[INFO] Creating time series embeddings...")

X, y = {}, {}

for k, df_ts in datasets_df_ts.items():
    print(f"  Processing {k}...")
    df_static = datasets_df_static[k]
    X[k], y[k] = genTSembeddings(df_ts, df_static, CONT_VARS, COL_LABELS)

print("✓ Time series embeddings created")

In [ ]:
datasets_df_ts.keys()

# Fidelity

## icd codes (Table 1)

### n-gram single-visit

In [ ]:
# Compute n-grams for ICD codes
# Reference: https://medium.com/@abhishekjainindore24/n-grams-in-nlp-a7c05c1aff12

codes_ngram = {}
codes = {}

for n in range(1, 4):
    print(f"Computing {n}-grams...")
    codes_ngram[n] = {}
    codes[n] = {}
    
    for k, dataset in datasets_icd.items():
        temp = [p["visits"][0] for p in dataset]
        codes_ngram[n][k] = compute_ngram_dict(temp, n)
        
        # Compute probabilities
        if n == 1:
            codes[n][k] = {
                word: count / N_WORDS 
                for word, count in codes_ngram[n][k].items()
            }
        else:
            codes[n][k] = {
                comb: count / codes_ngram[n-1][k][comb[:(n-1)]] 
                for comb, count in codes_ngram[n][k].items()
            }

print("✓ N-grams computed")

In [ ]:
# Analyze n-gram fidelity
dict_ngram = {}
df_ngram = pd.DataFrame(columns=['1-gram', '2-gram', '3-gram'], index=RUN_NAMES_ICD)

for n in range(1, 4):
    print(f"\nAnalyzing {n}-grams...")
    fig = go.Figure()
    maxVal, minVal = 0.5, 0.0

    def add_trace(code1, code2, name):
        """Add scatter trace for n-gram comparison."""
        global maxVal, minVal
        
        common_codes = list(set(code1.keys()).intersection(set(code2.keys())))
        n_unique = len(common_codes)
        
        if n_unique == 0:
            print(f"  ERROR: No common {n}-grams for Train and {name}")
            return np.nan
            
        print(f"  Common {n}-grams Train-{name}: {n_unique}/{len(code1)}")
        
        # Select top N by frequency
        common_codes = sorted(common_codes, key=lambda x: code1[x], reverse=True)[:N_TOP_NGRAMS]
        
        data1 = [code1[x] for x in common_codes]
        data2 = [code2[x] for x in common_codes]
        
        # Compute correlation metric
        metric_val, _ = pearsonr(data1, data2)
        print(f"  Pearson correlation: {metric_val:.3f}")
        
        # Create hover texts
        L1 = len(datasets_icd['train'])
        L2 = len(datasets_icd['test']) if name == 'test' else len(datasets_icd[name])
        hovertexts = [
            f"{x}: ({int(freq1*L1)}, {int(freq2*L2)})" 
            for x, freq1, freq2 in zip(common_codes, data1, data2)
        ]
        
        maxVal = max(data1)
        minVal = min(data1)
        
        fig.add_trace(go.Scatter(
            x=data1, y=data2, mode='markers', 
            name=f'train-{name}', hovertext=hovertexts
        ))
        
        return metric_val
    
    # Compare with test set
    r2_value_test = add_trace(codes[n]['train'], codes[n]['test'], 'test')
    dict_ngram[f'test-{n}'] = r2_value_test
    df_ngram.loc['test', f'{n}-gram'] = r2_value_test
    
    # Compare with synthetic datasets
    for k in RUN_NAMES_ICD:
        r2_value_syn = add_trace(codes[n]['train'], codes[n][k], k)
        dict_ngram[f'{k}{n}'] = r2_value_syn
        df_ngram.loc[k, f'{n}-gram'] = r2_value_syn
    
    # Add diagonal reference line
    fig.add_trace(go.Scatter(
        x=[minVal, maxVal*1.1], y=[minVal, maxVal*1.1],
        mode="lines", line=dict(color="red"), name="y=x"
    ))
    
    fig.update_layout(
        xaxis_title="Train", template="plotly", title=f"{n}-gram"
    )
    
    fig.write_html(f"{OUTPUT_DIR}/{n}-gram.html")

df_ngram.round(3)

### sequential bi-gram

What is it?

patients might have multiple visits (hospital admission). If we have code1 in the first visit and code2 in the second visit, we can create a bi-gram code1_code2. This is a sequential bi-gram.

In [ ]:
# Compute sequential bi-grams
print("Computing sequential bi-grams...")

codes_seq = {}
n = 2
codes_seq[n] = {}

for k, dataset in datasets_icd.items():
    temp = []
    for p in dataset:
        if len(p['visits']) > 1:
            for i in range(len(p["visits"]) - 1):
                temp.append((p["visits"][i], p["visits"][i+1]))
    
    print(f"  {k}: {len(dataset)} patients, {len(temp)} sequential pairs")
    codes_seq[n][k] = compute_bigram_seq(temp)

# Fix: pehr uses same sequential bigrams as HALO
codes_seq[2]['pehr'] = codes_seq[2]['HALO']

print("✓ Sequential bi-grams computed")

In [ ]:
dict_ngram = {}


fig = go.Figure()


maxVal=0.5
minVal=0.0

def add_trace(code1, code2, name):
    global maxVal, minVal
    
    common_codes = list(set(code1.keys()).intersection(set(code2.keys())))
    n_unique = len(common_codes)
    print(f"Number of common {n}-grams for Train and {name}: {n_unique}/{len(code1)}")
    # randomly select 1000 codes with highest frequency and seed=42
    common_codes = sorted(common_codes, key=lambda x: code1[x], reverse=True)[:N_TOP]
    
    data1 = [code1[x] for x in common_codes]
    data2 = [code2[x] for x in common_codes]
    
    metric_val = r2_score(data1, data2)
    metric_val,_ = pearsonr(data1, data2)

    print(f"R2 value for {name}: {metric_val:.2f}")
    # data1 = [np.log(code1[x]) for x in common_codes]
    # data2 = [np.log(code2[x]) for x in common_codes]
    L1 = len(datasets_icd['train'])
    L2 = len(datasets_icd['test']) if name == 'test' else len(datasets_icd[name])
    hovertexts = [f"{x} : {int(freq1*L1),int(freq2*L2)} " for x, freq1,freq2 in zip(common_codes, data1,data2)]
    maxVal = max(data1)
    minVal = min(data1)

    _ = fig.add_trace(go.Scatter(x=data1, y=data2, mode='markers', name=f'train-{name}', hovertext=hovertexts))    

    return metric_val
r2_value_test = add_trace(codes_seq[n]['train'], codes_seq[n]['test'], 'test')
dict_ngram[f'test-{n}'] = r2_value_test
df_ngram.loc['test', 'bi-gram-seq'] = r2_value_test

for k in RUN_NAMES_ICD:
    r2_value_syn = add_trace(codes_seq[n]['train'], codes_seq[n][k], k)
    dict_ngram[f'{k}{n}'] = r2_value_syn
    df_ngram.loc[k, 'bi-gram-seq'] = r2_value_syn
    # print(f"R2 value for {k}: {r2_value_syn:.2f}")
# r2_value_syn = add_trace(codes[n]['train'], codes[n][RUN_NAME], RUN_NAME)


_ = fig.add_trace(
    go.Scatter(
        x=[minVal, maxVal*1.1],
        y=[minVal, maxVal*1.1],
        mode="lines",
        line=dict(color="red"),
        name="y=x",
    )
)

# add title



_ = fig.update_layout(
    xaxis_title="Train",
    # yaxis_title="Test",
    template="plotly",
    title=f"{n}-gram  ", #  |   R2: {r2_value_syn:.2f}, {r2_value_test:.2f}
)

# _ = fig.show()

# save figure
fig.write_html(f"{OUTPUT_DIR}/{n}-gram-seq.html")


df_ngram

In [ ]:
df_ngram

# round to 3 decimal points
df_ngram.astype(float).round(3).dropna(axis=1)

# save the results
df_ngram.astype(float).round(3).dropna(axis=1).to_csv(f"{OUTPUT_DIR}/fid-icd-ngram.csv")

### Qualitative

some additional results that are not in the paper

In [ ]:
# distribution of number of stays
# PromptEHR can only handle one stay per patient

n_stays = {}


fig = go.Figure()
for k, dataset in datasets_icd.items():
    n_stays[k] = [len(x['visits']) for x in dataset]
    _ = fig.add_trace(go.Histogram(x=n_stays[k], histnorm='probability',name=k))
    
_ = fig.update_layout(barmode='group')
_ = fig.update_traces(opacity=0.75)
_ = fig.update_layout(
    xaxis_title_text='Number of Admissions',
    yaxis_title_text='Probability',
    # title_text='Number of stays distribution'
)
_ = fig.update_xaxes(range=[0, 6])



_ = fig.show()
# save figure
fig.write_html(f"{OUTPUT_DIR}/N_Adms.html")



In [ ]:
# Distribution of ICD codes per admission
print("Analyzing ICD code distribution per admission...")

icd_lens = {}
jsd_results = []
avg_results = []
std_results = []

for n_adms in range(4):
    fig = go.Figure()
    dict_jsd = {}
    dict_avg = {}
    dict_std = {}

    for k, dataset in datasets_icd.items():
        icd_lens[k] = [
            len(p["visits"][n_adms]) 
            for p in dataset if len(p['visits']) > n_adms
        ]
        
        if len(icd_lens[k]) > 5:
            dict_avg[k] = np.mean(icd_lens[k])
            dict_std[k] = np.std(icd_lens[k])
        else:
            dict_avg[k] = np.nan
            dict_std[k] = np.nan
        
        fig.add_trace(go.Histogram(
            x=icd_lens[k], xbins=dict(size=5), 
            histnorm='probability', name=k
        ))

        if k != 'train':
            dict_jsd[k] = compute_jsd(icd_lens['train'], icd_lens[k])

    jsd_results.append(dict_jsd)
    avg_results.append(dict_avg)
    std_results.append(dict_std)

    fig.update_layout(
        barmode='overlay',
        xaxis_title_text=f'Number of ICD codes in admission {n_adms+1}',
        yaxis_title_text='Probability'
    )
    fig.update_traces(opacity=0.50)
    
    fig.show()
    fig.write_html(f"{OUTPUT_DIR}/N_ICDs_adm{n_adms}.html")

print("✓ ICD distribution analysis complete")

In [ ]:
# comparing the distriutions between train and other datasets

pd.DataFrame(jsd_results)

pd.DataFrame(avg_results)

pd.DataFrame(std_results)

pd.DataFrame(avg_results).round(1).astype(str) + " (" + pd.DataFrame(std_results).round(1).astype(str) + ")"

In [ ]:
print("jsd_results, low is better")
pd.DataFrame(jsd_results)

# print("avg_results, low is better")
# pd.DataFrame(avg_results)

# pd.DataFrame(std_results)

pd.DataFrame(avg_results).round(1).astype(str) + " (" + pd.DataFrame(std_results).round(1).astype(str) + ")"

## time series

### Corr Confusion Matrix (Fig 2)

In [ ]:
# Configuration for correlation analysis
LABEL_MAP = {
    0: '0',    # high-neg
    1: '1',    # medium-neg
    2: '2',    # low
    3: '3',    # medium-pos
    4: '4',    # high-pos
    100: 'nan' # missing/NaN correlations
}

SELECTED_LABELS = [0, 1, 2, 3, 4]
MODE_CORR = 'default'  # 'normalized' or 'default'

In [ ]:
# Plot correlation confusion matrices
print("Analyzing temporal correlations...")

conf_mats = {}  # Stores confusion matrices
corr_tcd = {}   # Stores Temporal Correlation Difference

for k in ['test'] + RUN_NAMES:
    print(f"\nProcessing {k}...")
    if k not in datasets_df_ts:
        print(f"  Skipping {k}, not found")
        continue

    conf_mats[k], corr_tcd[k] = plot_corr3(
        datasets_df_ts['train'], datasets_df_ts[k], CONT_VARS, 
        corr_th=0.0, corr_method='ffill'
    )

    # Normalize if needed
    if MODE_CORR == 'normalized':
        conf_normalized = conf_mats[k] / conf_mats[k].sum(axis=1)[:, None]
        conf_normalized[(conf_normalized < 0.01) & (conf_normalized > 0)] = 0.01
        conf_normalized = np.round(conf_normalized, 2)
    elif MODE_CORR == 'default':
        conf_normalized = conf_mats[k]

    # Plot confusion matrix
    print(f"  Temporal Correlation Difference: {corr_tcd[k]:.4f}")
    disp = ConfusionMatrixDisplay(
        confusion_matrix=conf_normalized, 
        display_labels=[LABEL_MAP[i] for i in SELECTED_LABELS]
    )
    disp.plot(xticks_rotation=45)
    plt.show()
    
    plt.savefig(f"{OUTPUT_DIR}/corr_{k}.png")
    plt.close()

print("✓ Correlation analysis complete")

### PRDC Metrics (Table 2)


In [ ]:
# Compute PRDC metrics
print("Computing PRDC metrics (this may take some time)...")

Xy = {}
LL = 4 * len(CONT_VARS)  # min/max/mean/std for each continuous variable

all_metrics = []

for random_state in [42, 43, 44]:
    print(f"\nRandom state: {random_state}")
    metrics_synth = {}

    # Prepare data
    for k in datasets_df_ts.keys():
        Xy[k] = X[k].fillna(0).iloc[:, :LL]
        Xy[k] = Xy[k].sample(N_SAMPLES_UTILITY, random_state=random_state, replace=False)
        Xy[k] = Xy[k] + np.random.normal(0, 0.00001, Xy[k].shape)

    # Compute metrics
    for k in [key for key in datasets_df_ts.keys() if key != 'train']:
        print(f"  Evaluating {k}...")
        metrics_synth[k] = compute_synthcity2(Xy['train'], Xy[k])

    all_metrics.append(metrics_synth)

print("✓ PRDC metrics computed")

In [ ]:
# Create results table for PRDC metrics
metric_names = list(pd.DataFrame(all_metrics[0]).index)
columns = list(pd.DataFrame(all_metrics[0]).columns)

mat = np.stack([pd.DataFrame(m).values for m in all_metrics])

# Compute mean and std
df_mean = pd.DataFrame(
    np.mean(mat, axis=0), columns=columns, index=metric_names
).round(3).T

df_std = pd.DataFrame(
    np.std(mat, axis=0), columns=columns, index=metric_names
).round(3).T

# Combine: mean (std)
df = df_mean.astype(str) + " (" + df_std.astype(str) + ")"
df

In [ ]:
# remove unwanted columns
if 'privacy.identifiability_score.score' in df.columns:
    df = df.drop(columns=['privacy.identifiability_score.score'])
if 'privacy.identifiability_score.score_OC' in df.columns:
    df = df.drop(columns=['privacy.identifiability_score.score_OC'])

# adding the temporal correlation difference
corr_series = pd.Series(corr_tcd)
df['TCD'] = corr_series.round(3).astype(str)

df

# save dataframe to csv
df.to_csv(f"{OUTPUT_DIR}/fid-ts.csv")

### PW Missingness (Fig 3)

In [ ]:

occ_mat={}


for i,k in  enumerate([key for key in datasets_df_ts.keys() if key != 'train']):
    print(k)

    df = datasets_df_ts[k][CONT_VARS]

    mat = df.notnull().astype(int).values

    # co-occurrence matrix
    c = mat.T @ mat

    # normalize
    x = mat.sum(axis=0)[None,:] + mat.sum(axis=0)[:,None] # sum of rows and columns
    c = np.round(c / x*100,2)

    # set diagonal and upper triangular to nan
    np.fill_diagonal(c, np.nan)
    c[np.triu_indices_from(c, 1)] = np.nan
    
    occ_mat[k] = c

    # plot heatmap
    fig= go.Figure()
    _ = fig.add_trace(go.Heatmap(z=c[::-1], colorscale='Viridis', zmin=0, zmax=50)) # x=cont_vars, y=cont_vars[::-1],
    

    _ = fig.update_layout(
        title_text=f"Co-occurrence matrix",
        template="plotly",
        height=400,
        width=400,
    )

    fig.show()

    print(f"MSE between test and {k}: {np.nanmean((occ_mat['test'] - occ_mat[k])**2):.2f}")
    
    # save figure as png
    # pio.write_image(fig, f"{OUTPUT_DIR}/occ-{k}.png")

    # save figure as html
    fig.write_html(f"{OUTPUT_DIR}/occ-{k}.html")
    






    


In [ ]:
for k in occ_mat.keys():
# print mse between test and k (set nans to 0)
    
    print(f"MSE between test and {k}: {np.nanmean((occ_mat['test'] - occ_mat[k])**2):.2f}")

### t-SNE

In [ ]:
# Prepare data for t-SNE visualization
print("Preparing t-SNE visualization...")

data_tsne = {}

for k in datasets_df_ts.keys():
    print(f"  {k}")
    data_tsne[k] = X[k].fillna(0).values

fig_tsne = plot_tsne(data_tsne, N=3000)
fig_tsne.show()

print("✓ t-SNE visualization created")

In [ ]:
# Save t-SNE visualization
fig_tsne.write_html(f"{OUTPUT_DIR}/tsne.html")
print(f"✓ t-SNE saved to {OUTPUT_DIR}/tsne.html")

## Covars&Labels

These results are not available in the paper.

In [ ]:



# for gender

fig = go.Figure()

for k, df in datasets_df_static.items():
    
    _ = fig.add_trace(go.Histogram(x=df['Gender'], histnorm='probability', name=k))

_ = fig.update_layout(barmode='overlay')
_ = fig.update_traces(opacity=0.75)

_ = fig.show()

# save
fig.write_html(f"{OUTPUT_DIR}/gender.html")

# for age

fig = go.Figure()

for k, df in datasets_df_static.items():

    _ = fig.add_trace(go.Histogram(x=df['Age'],xbins=dict(size=5) ,histnorm='probability', name=k))

_ = fig.update_layout(barmode='overlay')
_ = fig.update_traces(opacity=0.75)

_ = fig.show()

# save
fig.write_html(f"{OUTPUT_DIR}/age.html")

In [ ]:
# distribution of label probs for the first admission of each patient
# n_labels = len(datasets['train'][0]['labels_phe'][0])

col_labels2 = [f'label_phe_{i}' for i in range(25)] + ['label_ihm']
n_labels = len(col_labels2)
label_probs = {}
fig = go.Figure()
for k, dataset in datasets_df_static.items():


    try:
        label_probs[k] = []
        for c in col_labels2:
            if c in dataset.columns:
                label_probs[k].append(dataset[c].mean())
            else:
                label_probs[k].append(np.nan)
        # label_probs[k] = dataset[col_labels2].mean().values
        

        # label_probs[k] = [
        #     len([p for p in dataset if p["labels_phe"][0][i] == 1]) / len(dataset) for i in range(n_labels)
        # ]
        _ = fig.add_trace(go.Bar(x=col_labels2, y=label_probs[k], name=k))
    except:
        print(f"{k} not found")
    


_ = fig.update_layout(barmode='group')
_ = fig.update_traces(opacity=0.75)

_ = fig.show()

# save figure
fig.write_html(f"{OUTPUT_DIR}/label_probs.html")

# save
fig.write_html(f"{OUTPUT_DIR}/label_probs.html")

# Utility (Tables 3,4)

In [ ]:
# Configuration for utility analysis
DOWNSTREAM_TASK = 'ihm'  # 'ihm' (in-hospital mortality) or 'phe' (phenotyping)

metrics_utility = {}

In [ ]:
# Define train/synthetic mixing ratios
train_fake_ratio = [
    (0, 1),      # 0% real, 100% synthetic (TSTR)
    (0.1, 1), (0.1, 0),  # 10% real with/without synthetic
    (0.2, 1), (0.2, 0),  # 20% real with/without synthetic
    (0.5, 1), (0.5, 0),  # 50% real with/without synthetic
    (1, 1), (1, 0),      # 100% real with/without synthetic (TRTR)
]

print(f"Utility evaluation with {len(train_fake_ratio)} configurations")

In [ ]:
# Train XGBoost models for downstream tasks
print("Training XGBoost models for utility evaluation...")

for k in tqdm(['val'] + RUN_NAMES, leave=False):
    print(f"\n  Evaluating {k}...")
    metrics_utility[k] = compute_utility2(
        X['train'].fillna(0), y['train'],
        X['test'].fillna(0), y['test'],
        X[k].fillna(0), y[k],
        train_fake_ratio=train_fake_ratio
    )

print("✓ Utility evaluation complete")

In [ ]:
metrics_utility.keys()
metrics_utility[list(metrics_utility.keys())[0]].keys()

all_metrics = list(metrics_utility[list(metrics_utility.keys())[0]][(0,1)].keys())
all_metrics

In [ ]:
metrics_utility.keys()



for metric in all_metrics:
    dict_metric = {}
    sample_run = metrics_utility[list(metrics_utility.keys())[0]]
    fig = go.Figure()
    curve_train_only = {k[0]:v[metric] for k,v in sample_run.items() if k in [(0.1,0), (0.2,0), (0.5,0), (1,0)]}

    curve_train_only
    dict_metric['train only'] = [0] + list(curve_train_only.values())

    _ = fig.add_trace(go.Scatter(x=list(curve_train_only.keys()), y=list(curve_train_only.values()), mode='lines+markers', name='train only',line= dict(dash='dash', color='black')))

    for run in ['val']+RUN_NAMES[:]:
        curve_current_run = {k[0]:v[metric] for k,v in metrics_utility[run].items() if k in [(0,1), (0.1,1), (0.2,1), (0.5,1), (1,1)]}
        dict_metric[run] = list(curve_current_run.values())


        _ = fig.add_trace(go.Scatter(x=list(curve_current_run.keys()), y=list(curve_current_run.values()), mode='lines+markers', name=run))


    _ = fig.update_layout(
        title=f"{metric}",
    )

    fig.show()

    print(metric)
    pd.DataFrame(dict_metric).T.round(3)


    # save dataframe
    pd.DataFrame(dict_metric).T.round(3).to_csv(f"{OUTPUT_DIR}/utility-{DOWNSTREAM_TASK}-{metric}.csv")



# Privacy (Table 5)

## icd codes

In [ ]:
# Extract ICD codes for privacy analysis
print("Preparing ICD codes for privacy analysis...")

codes = {}

for k, dataset in datasets_icd.items():
    codes[k] = [v for p in dataset for v in p["visits"] if len(v) > 0]
    # Shuffle with fixed seed for reproducibility
    random.seed(42)
    random.shuffle(codes[k])
    print(f"  {k}: {len(codes[k])} visit sequences")

print("✓ ICD codes prepared")

In [ ]:
# Compute pairwise distance matrices for privacy analysis
print(f"Computing distance matrices (N={N_SAMPLES_PRIVACY})...")

D_syn_train = {}
D_syn_test = {}

for k in RUN_NAMES_ICD:
    print(f"\n{k}:")
    cache_train = f"temp/D_{k}_train.npy"
    cache_test = f"temp/D_{k}_test.npy"
    
    if os.path.exists(cache_train):
        D_syn_train[k] = np.load(cache_train)
        D_syn_test[k] = np.load(cache_test)
        print("  Loaded from cache")
    else:
        D_syn_train[k] = pairwise_hamming_distance(
            codes[k][:N_SAMPLES_PRIVACY], 
            codes['train'][:N_SAMPLES_PRIVACY]
        )
        D_syn_test[k] = pairwise_hamming_distance(
            codes[k][:N_SAMPLES_PRIVACY], 
            codes['test'][:N_SAMPLES_PRIVACY]
        )
        
        # Save to cache
        os.makedirs("temp", exist_ok=True)
        np.save(cache_train, D_syn_train[k])
        np.save(cache_test, D_syn_test[k])
        print("  Computed and cached")

print("✓ Distance matrices ready")

In [ ]:
# Analyze privacy metrics for ICD codes
print("Computing privacy metrics for ICD codes...")

dict_jsd = {}
dict_wd = {}
dict_auroc = {}
dict_aa_train = {}

for k in RUN_NAMES_ICD:
    print(f"\n{k}:")
    fig = go.Figure()
    
    min_d_syn_train = D_syn_train[k].min(axis=1)
    min_d_syn_test = D_syn_test[k].min(axis=1)
    
    print(f"  Distance comparison accuracy: {comp_acc(min_d_syn_train, min_d_syn_test):.3f}")
    dict_aa_train[k] = comp_acc(min_d_syn_train, min_d_syn_train)
    
    # Fit KDE and compute metrics
    kde_train = gaussian_kde(min_d_syn_train)
    kde_test = gaussian_kde(min_d_syn_test)
    
    max_val = max(max(min_d_syn_train), max(min_d_syn_test))
    x = np.linspace(0, max_val, 1000)
    y_train = kde_train(x)
    y_test = kde_test(x)
    
    dict_jsd[k] = jensenshannon(y_train, y_test)
    dict_wd[k] = wasserstein_distance(y_train, y_test)
    dict_auroc[k] = roc_auc_score(
        np.concatenate([np.ones_like(min_d_syn_train), np.zeros_like(min_d_syn_test)]),
        np.concatenate([
            [kde_train(x) for x in min_d_syn_train],
            [kde_test(x) for x in min_d_syn_test]
        ])
    )
    
    print(f"  JSD: {dict_jsd[k]:.4f}, WD: {dict_wd[k]:.4f}, AUROC: {dict_auroc[k]:.4f}")
    
    # Plot histograms
    _ = fig.add_trace(go.Histogram(
        x=min_d_syn_train, histnorm='probability', name=f'{k}-train'
    ))
    _ = fig.add_trace(go.Histogram(
        x=min_d_syn_test, histnorm='probability', name=f'{k}-test'
    ))
    
    _ = fig.update_layout(barmode='overlay')
    _ = fig.update_traces(opacity=0.75)
    fig.show()

print("✓ ICD privacy analysis complete")

In [ ]:
# Display and save ICD privacy results
df_privacy_icd = pd.DataFrame({
    'JSD': dict_jsd, 
    'WD': dict_wd, 
    'AUROC': dict_auroc
}).round(4)

df_privacy_icd.to_csv(f"{OUTPUT_DIR}/priv-icd.csv")
print(f"✓ Privacy results saved to {OUTPUT_DIR}/priv-icd.csv")

df_privacy_icd

## time series

In [ ]:
# Compute privacy metrics for time series
print("Computing privacy metrics for time series...")

Xy = {}
LL = 4 * len(CONT_VARS)  # min/max/mean/std for each continuous variable

all_metrics_mia = []
all_metrics_nnaa = []

for random_state in tqdm([42, 43, 44], desc="Random states"):
    metrics_mia = {}
    metrics_nnaa = {}

    # Prepare data
    for k in ['train', 'test'] + RUN_NAMES:
        Xy[k] = pd.concat([X[k].fillna(0).iloc[:, :LL], y[k]], axis=1)
        Xy[k] = Xy[k].sample(N_SAMPLES_PRIVACY, random_state=random_state, replace=False)
        Xy[k] = Xy[k] + np.random.normal(0, 0.00001, Xy[k].shape)

    # Compute metrics
    for k in tqdm(RUN_NAMES, desc="Models", leave=False):
        REAL = Xy['train'].values
        FAKE = Xy[k].values
        TEST = Xy['test'].values

        metrics_mia[k] = compute_mia_knn(REAL, FAKE, TEST)
        metrics_nnaa[k] = compute_nnaa(REAL, FAKE, TEST)

    all_metrics_mia.append(metrics_mia)
    all_metrics_nnaa.append(metrics_nnaa)

print("✓ Time series privacy metrics computed")

In [ ]:
# Format and save time series privacy results
df_mia = prettify_metrics(all_metrics_mia)
df_mia.to_csv(f"{OUTPUT_DIR}/privacy-ts-mia.csv")
print(f"✓ MIA results saved to {OUTPUT_DIR}/privacy-ts-mia.csv")

df_nnaa = prettify_metrics(all_metrics_nnaa)
df_nnaa.to_csv(f"{OUTPUT_DIR}/privacy-ts-nnaa.csv")
print(f"✓ NNAA results saved to {OUTPUT_DIR}/privacy-ts-nnaa.csv")

# Display combined results
df_privacy_ts = pd.concat([df_mia, df_nnaa], axis=1)
df_privacy_ts